# 04 · Maximum lime doses by state

One row per stated upper limit on a lime application, compiled from the state extension sources. Output: `data/max_dose.csv`.


In [1]:
import numpy as np, pandas as pd
from pathlib import Path

URL = {
    "U0": "https://www.udel.edu/content/dam/udelImages/canr/pdfs/extension/factsheets/Sikora-Buffer-Tables.pdf",
    "U1": "https://www.dcr.virginia.gov/document/standardsandcriteria.pdf",
    "U2": "https://extension.umd.edu/sites/extension.umd.edu/files/2021-04/SFM-5.pdf",
    "U3": "https://umaine.edu/soiltestinglab/wp-content/uploads/sites/227/2016/07/commpam.pdf",
    "U4": "https://umaine.edu/soiltestinglab/wp-content/uploads/sites/227/2016/07/handbook.pdf",
    "U5": "https://plantsoilcenter.cahnr.uconn.edu/interpreting-your-results/interpretation-of-soil-test-results/",
    "U6": "https://www.umass.edu/agriculture-food-environment/crops-dairy-livestock-equine/fact-sheets/soil-acidity-liming",
    "U7": "https://www.umass.edu/agriculture-food-environment/turf/fact-sheets/soil-ph-liming",
    "U8": "https://www.umass.edu/agriculture-food-environment/soil-plant-nutrient-testing-laboratory/fact-sheets/timing-of-lime-fertilizer-applications",
    "U9": "http://extension.cropsciences.illinois.edu/handbook/pdfs/chapter08.pdf",
    "U10": "https://www.extension.purdue.edu/extmedia/AY/AY-9-32.html",
    "U11": "https://bookstore.ksre.ksu.edu/pubs/mf2586.pdf",
    "U12": "https://extension.oregonstate.edu/sites/extd8/files/documents/em9057.pdf",
    "U13": "https://extension.oregonstate.edu/sites/extd8/files/documents/em9060.pdf",
    "U14": "https://cropsandsoils.extension.wisc.edu/nutrient-application-guidelines-for-field-vegetable-and-fruit-crops-in-wisconsin-a2809/soil-ph-and-lime-requirement/",
    "U15": "https://idahodocs.contentdm.oclc.org/digital/api/collection/p16293coll6/id/897/download",
    "U16": "https://www.uaex.uada.edu/publications/PDF/FSA-6134.pdf",
    "U17": "https://extension.missouri.edu/publications/g9102",
    "U18": "https://publications.mgcafe.uky.edu/sites/publications.ca.uky.edu/files/AGR-1-Lime-and-Fertilizer-Recommendations",
}

rows = [
    # state, mixdepth_cm, max_dose, dose_units, material_basis_verbatim, basis_kind, cce_factor, dose_note, source_url, source_note
    ('DE', 15.24, 4, 'short_ton_acre', 'agricultural lime (67 % ECCC)', 'ECCC', 0.67, 'max single application when incorporated with tillage; split applications above this', URL["U0"], 'text below the buffer tables, p. 1; search "maximum single application rate"'),
    ('DE', None, 2, 'short_ton_acre', 'agricultural lime (67 % ECCC)', 'ECCC', 0.67, 'max single application when surface applied; split above this', URL["U0"], 'same sentence as the 4 t/ac limit, p. 1; search "2 ton/ac if surface applying"'),

    ('VA', None, 2, 'short_ton_acre', 'limestone', 'CCE', 0.9, 'max single application in no-till (surface, not incorporated); otherwise half one year, half the next', URL["U1"], 'liming section, p. 52; search "single applications should be limited to no more than 2 tons"'),
    ('VA', 15.24, 2, 'short_ton_acre', 'limestone', 'CCE', 0.9, 'split threshold, conventional tillage: above 2 t/ac apply half disked in and plowed under, half disked in', URL["U1"], 'liming section, p. 52; search "Applications of more than 2 tons per acre"'),

    ('MD', None, 1500, 'lb_acre', 'total oxides (CaO + MgO)', 'oxide_sum', 2.07, 'annual max (lb/acre/yr) for continuous no-till and topdressed hay, pasture, established alfalfa; retest within a year', URL["U2"], 'text after the lime tables, p. 5; search "should not exceed 1,500 pounds total oxides"'),
    ('MD', 20.32, 4000, 'lb_acre', 'total oxides (CaO + MgO)', 'oxide_sum', 2.07, 'max single application, Table 2 (agronomic crops, target 6.5), loam soils; marked ** in the table. Table 3 (alfalfa, 7.0): 4,500 loam; Table 4 (tobacco, 5.6): 750', URL["U2"], 'Table 2 footnote **, p. 5; search "maximum amount that should be applied in one application"'),
    ('MD', 20.32, 4500, 'lb_acre', 'total oxides (CaO + MgO)', 'oxide_sum', 2.07, 'as above, silt loam and clay loam soils (Table 2). Table 3 (alfalfa, 7.0): 5,000', URL["U2"], 'Table 2 footnote **, p. 5; search "maximum amount that should be applied in one application"'),

    ('ME', 16.93, 6000, 'lb_acre', 'lime rated at 100% calcium carbonate equivalence (CCE)', 'CCE', 1, 'split threshold: above this, split over two or more years (commercial crops)', URL["U3"], 'lime recommendation section, p. 4; search "more than 6000 pounds per acre"'),
    ('ME', None, 2000, 'lb_acre', 'lime rated at 100% calcium carbonate equivalence (CCE)', 'CCE', 1, 'annual max when topdressed (lb/acre in any one year)', URL["U3"], 'lime recommendation section, p. 4; search "limited to 2000 pounds per acre"'),
    ('ME', None, 4000, 'lb_acre', 'lime (handbook basis: 100% CaCO3)', 'CCE', 1, 'annual max topdress, all topdressed crops (lb/acre in any one year)', URL["U4"], 'crop recommendation pages; search "Limit lime topdress to 4000 pounds"'),
    ('ME', None, 100, 'lb_1000sqft', 'lime (handbook basis: 100% CaCO3)', 'CCE', 1, 'annual max topdress, turf (lb/1000 sq ft in any one year)', URL["U4"], 'turf recommendation page; search "Limit lime topdress to 100 pounds"'),

    ('CT', None, 75, 'lb_1000sqft', 'ground limestone', 'unspecified', None, 'max surface application at one time (range 50-75 lb/1000 sq ft) unless tilled in; reapply at 1-6 month intervals', URL["U5"], 'limestone section; search "Unless the limestone is to be tilled in"'),

    ('MA', None, 4, 'short_ton_acre', 'limestone', 'unspecified', None, 'split threshold, field crops: at 4 t/acre or more, two applications 6 months apart', URL["U6"], 'liming section; search "4 tons or more per acre"'),
    ('MA', None, 70, 'lb_1000sqft', 'limestone', 'unspecified', None, 'max single application, established turf (≈1.5 t/acre)', URL["U7"], 'liming section; search "should not exceed 70 pounds per 1000"'),
    ('MA', None, 25, 'lb_1000sqft', 'limestone', 'unspecified', None, 'max single application, athletic fields and putting greens on sand rootzones (≈0.5 t/acre)', URL["U7"], 'liming section; search "sand rootzones should not exceed 25 pounds"'),
    ('MA', None, 50, 'lb_1000sqft', 'limestone', 'unspecified', None, 'max single application, existing gardens, trees and shrubs; printed as 5 lb per 100 sq ft', URL["U8"], 'list under \'To avoid damaging existing plantings\'; search "5 lbs. per 100 square feet"'),
    ('MA', None, 50, 'lb_1000sqft', 'limestone', 'unspecified', None, 'max single application, existing lawns, turf and commercial nursery crops', URL["U8"], 'same list; search "50 lbs. per 1,000 square feet"'),
    ('MA', None, 1, 'short_ton_acre', 'limestone', 'unspecified', None, 'max single application, existing commercial crop plantings (new plantings: full rate at once)', URL["U8"], 'same list; search "1 ton per acre for commercial crops"'),

    ('IL', 22.86, 6, 'short_ton_acre', 'limestone', 'CCE', 0.9, 'max single application, general cropland (incorporated); if cost matters split ~2/3 now, rest in 3-4 years', URL["U9"], 'liming section, p. 9; search "up to 6 tons per acre may be applied"'),
    ('IL', 22.86, 5, 'short_ton_acre', 'limestone', 'CCE', 0.9, 'split threshold, hay and pasture seeding: half before primary tillage, half before secondary', URL["U9"], 'liming section, p. 9; search "exceed 5 tons per acre"'),

    ('IN', 20.32, 8, 'short_ton_acre', 'agricultural ground limestone with a neutralizing value of 90 percent', 'CCE', 0.9, 'seasonal max (tons in one season); Tri-State recommendations also used for Michigan', URL["U10"], 'liming section; search "more than 8 tons of lime in one season"'),
    ('IN', 20.32, 4, 'short_ton_acre', 'agricultural ground limestone with a neutralizing value of 90 percent', 'CCE', 0.9, 'split threshold: above 4 t/acre apply half before and half after plowing', URL["U10"], 'liming section; search "exceeds 4 tons per acre"'),

    ('KS', 15.24, 10000, 'lb_acre', 'effective calcium carbonate (ECC)', 'ECC', 1, 'split threshold, all crops: above this apply half rate, incorporate, wait 12-18 months, retest', URL["U11"], 'lime recommendation equations, p. 10; search "10,000 lb ECC/a"'),

    ('OR', None, 2, 'short_ton_acre', 'lime', 'unspecified', None, 'max single application, acid-loving plants (blueberry, azalea, red maple)', URL["U12"], 'liming section, p. 6; search "no more than 2 t lime/a"'),
    ('OR', None, 2, 'short_ton_acre', 'lime', 'unspecified', None, 'max topdress application, perennial pasture and no-till', URL["U12"], 'surface application section, p. 6; search "Rates higher than 2 t/a are not warranted"'),
    ('OR', None, 2, 'short_ton_acre', '100-score lime', 'CCE', 1, 'max single application, very sandy irrigated soils (Columbia Basin)', URL["U13"], 'liming section, p. 2; search "should exceed 2 t/a of 100-score lime"'),

    ('WI', 17.78, 8, 'short_ton_acre', '60-69 NI lime', 'NI', 0.65, 'max single application, potato', URL["U14"], 'text below Table 5.1; search "should never exceed 8 tons/a"'),
    ('WI', 17.78, 12, 'short_ton_acre', '60-69 NI lime', 'NI', 0.65, 'max single application, all crops other than potato', URL["U14"], 'text below Table 5.1; search "12 tons/a for other crops"'),
    ('WI', 17.78, 1, 'short_ton_acre', '60-69 NI lime', 'NI', 0.65, 'sandy soils < 1% OM when the calculated requirement is < 1.5 t/a', URL["U14"], 'text below Table 5.1; search "only 1 ton/a of lime should be applied"'),
    ('WI', None, 1, 'short_ton_acre', '60-69 lime or 80-89 lime', 'NI', 0.65, 'topdress rate for fields in no-till > 5 years with surface (0-2 in) pH > 0.2 below target', URL["U14"], 'no-till paragraph; search "1 ton/a of topdressed"'),

    ('ID', None, 500, 'lb_acre', 'fluid (liquid) lime', 'unspecified', None, 'max for fluid lime applications', URL["U15"], 'CIS 787, fluid lime paragraph, p. 4; search "maximum of only 500 pounds of lime per acre"'),

    ('AR', None, 50, 'lb_1000sqft', 'lime', 'CCE', 0.915, 'max single application, home lawn and turf; split spring/fall above this', URL["U16"], 'lawn liming section, p. 1; search "more than 50 lb lime per 1,000"'),

    ('MO', None, 1500, 'lb_acre', 'pounds ENM (effective neutralizing material)', 'ENM', 2.5, 'max single application, no-till, regardless of soil requirement; apply more a year later', URL["U17"], 'no-till section; search "1,500 pounds ENM per acre"'),
    ('MO', None, 3, 'short_ton_acre', 'agricultural limestone', 'unspecified', None, 'max at one time when lime can only be topdressed (established alfalfa, very acid soil); remainder a year later', URL["U17"], 'topdressing paragraph; search "no more than 3 tons of agricultural limestone"'),

    ('KY', 15.24, 4, 'short_ton_acre', 'agricultural limestone (RNV basis)', 'RNV', 1, 'split threshold, cropland: above 4 t/A, half before and half after plowing', URL["U18"], 'liming section, p. 4-5; search "lime rates greater than 4 ton/A"'),
    ('KY', 15.24, 4, 'short_ton_acre', 'limestone', 'RNV', 1, 'split threshold, tobacco (or spring liming before transplanting)', URL["U18"], 'tobacco section; search "If more than 4 ton/ acre"'),
    ('KY', None, 100, 'lb_1000sqft', 'limestone', 'RNV', 1, 'max single application, turf maintenance (range 70-100 lb/1000 sq ft); further half rates at 3-6 month intervals', URL["U18"], 'turf section; search "more than 70 to 100 lb"'),
]

cols = ["state", "mixdepth_cm", "max_dose", "dose_units", "material_basis_verbatim", "basis_kind", "cce_factor",
        "dose_note", "source_url", "source_note"]
maxdose = pd.DataFrame(rows, columns=cols)
maxdose.to_csv(Path("../data") / "max_dose.csv", index=False)
print(f"{len(maxdose)} limits from {maxdose.state.nunique()} states")
maxdose[["state", "max_dose", "dose_units", "basis_kind", "dose_note"]]

37 limits from 15 states


,state,max_dose,dose_units,basis_kind,dose_note
0,DE,4,short_ton_acre,ECCC,max single application when incorporated with ...
1,DE,2,short_ton_acre,ECCC,max single application when surface applied; s...
2,VA,2,short_ton_acre,CCE,"max single application in no-till (surface, no..."
3,VA,2,short_ton_acre,CCE,"split threshold, conventional tillage: above 2..."
4,MD,1500,lb_acre,oxide_sum,annual max (lb/acre/yr) for continuous no-till...
5,MD,4000,lb_acre,oxide_sum,"max single application, Table 2 (agronomic cro..."
6,MD,4500,lb_acre,oxide_sum,"as above, silt loam and clay loam soils (Table..."
7,ME,6000,lb_acre,CCE,"split threshold: above this, split over two or..."
8,ME,2000,lb_acre,CCE,annual max when topdressed (lb/acre in any one...
9,ME,4000,lb_acre,CCE,"annual max topdress, all topdressed crops (lb/..."


## On a common basis

The same limits as kg CCE ha⁻¹ (blank where the source names no material basis).

In [2]:
LB_ACRE = {"lb_acre": 1.0, "short_ton_acre": 2000.0, "lb_1000sqft": 43.56}
maxdose["max_kgCCE_ha"] = (maxdose.max_dose * maxdose.dose_units.map(LB_ACRE) * 1.12085 * maxdose.cce_factor).round(-1)
maxdose.sort_values("max_kgCCE_ha", ascending=False)[["state", "max_kgCCE_ha", "dose_note"]]

,state,max_kgCCE_ha,dose_note
27,WI,17490.0,"max single application, all crops other than p..."
20,IN,16140.0,seasonal max (tons in one season); Tri-State r...
18,IL,12110.0,"max single application, general cropland (inco..."
26,WI,11660.0,"max single application, potato"
22,KS,11210.0,"split threshold, all crops: above this apply h..."
6,MD,10440.0,"as above, silt loam and clay loam soils (Table..."
19,IL,10090.0,"split threshold, hay and pasture seeding: half..."
5,MD,9280.0,"max single application, Table 2 (agronomic cro..."
35,KY,8970.0,"split threshold, tobacco (or spring liming bef..."
34,KY,8970.0,"split threshold, cropland: above 4 t/A, half b..."
